<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7B_Despliegue_Elastic_Cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# S07B — Repetir el despliegue de Elasticsearch por tu cuenta

Este cuaderno queda como **recurso opcional de repetición**, no como segundo camino de la clase.

La sesión oficial se realiza en un único notebook:
https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_Elasticsearch_BM25_Compras_Claras.ipynb

Úsalo después si quieres practicar nuevamente la secuencia mínima:
**Project URL → API key → Python client → índice → bulk → búsqueda**.

Guía vigente:
https://jazaineam1.github.io/BigData2026/assets/tutoriales/s07-despliegue-elasticsearch.html

## Antes de empezar

La ruta principal del curso usa Elasticsearch Serverless y el corpus completo de S6. Este cuaderno histórico se conserva para repetición rápida.

Si estás cursando S07 ahora, vuelve a la **ruta única**:
https://jazaineam1.github.io/BigData2026/assets/tutoriales/s07-recursos.html

No pegues credenciales en celdas que luego subirás a GitHub.

## 1. Instalar cliente oficial de Elasticsearch

En Colab la instalación toma poco tiempo. Si estás en tu computador Windows, esto equivale a instalar la librería en tu ambiente Python.

In [ ]:
!pip -q install "elasticsearch>=9,<10" pandas

## 2. Cargar corpus de respaldo

Usamos el corpus de S7 publicado en el repositorio. Si tienes tu `s06_contexto_procesos.jsonl`, también puedes subirlo y reemplazar esta celda.

In [ ]:
import pandas as pd

URL_CORPUS = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s07_corpus_respaldo.jsonl'
corpus = pd.read_json(URL_CORPUS, lines=True)
corpus = corpus.fillna('')
corpus[['id_proceso','entidad','nombre_proceso','descripcion','url_secop']].head()

## 3. Conectar con Elastic Cloud

La forma más segura en clase es pedir los datos con `getpass()`. Así no quedan visibles en la salida del cuaderno.

In [ ]:
from getpass import getpass
from elasticsearch import Elasticsearch

ES_ENDPOINT = input('Endpoint de Elasticsearch (https://...): ').strip()
ES_API_KEY = getpass('API key de Elasticsearch: ').strip()

client = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=30)
info = client.info()
info

### Si esta celda falla

| Error | Lectura | Acción |
|---|---|---|
| 401/403 | credencial inválida o sin permisos | recrea API key |
| timeout | red o endpoint mal copiado | prueba endpoint y conexión |
| SSL/certificado | endpoint incompleto o sin `https` | copia el endpoint completo |

## 4. Definir mapping

Aquí decidimos qué campos se analizan como texto y cuáles se conservan como valores exactos.

In [ ]:
INDEX_NAME = 's07_compras_claras'

mapping = {
    'mappings': {
        'properties': {
            'id_proceso': {'type': 'keyword'},
            'entidad': {'type': 'keyword'},
            'modalidad': {'type': 'keyword'},
            'nombre_proceso': {'type': 'text', 'analyzer': 'spanish'},
            'descripcion': {'type': 'text', 'analyzer': 'spanish'},
            'url_secop': {'type': 'keyword', 'index': False}
        }
    }
}

if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
client.indices.create(index=INDEX_NAME, **mapping)
print('Índice creado:', INDEX_NAME)

## 5. Cargar documentos con bulk

`bulk` permite enviar muchos documentos de una vez. Para el curso el volumen es pequeño, pero el patrón es el mismo cuando crece.

In [ ]:
from elasticsearch.helpers import bulk

actions = []
for d in corpus.to_dict('records'):
    source = {
        'id_proceso': str(d.get('id_proceso','')),
        'entidad': str(d.get('entidad','')),
        'modalidad': str(d.get('modalidad','')),
        'nombre_proceso': str(d.get('nombre_proceso','')),
        'descripcion': str(d.get('descripcion','')),
        'url_secop': str(d.get('url_secop',''))
    }
    actions.append({'_index': INDEX_NAME, '_id': source['id_proceso'], '_source': source})

ok, errores = bulk(client, actions, refresh=True, raise_on_error=False)
print('Documentos cargados:', ok, '| errores:', len(errores))

## 6. Buscar con `multi_match`

Buscamos en `nombre_proceso` y `descripcion`. El `^2` le da más peso al nombre del proceso.

In [ ]:
consulta = 'mantenimiento aeronaves'

query = {
    'query': {
        'bool': {
            'must': [{
                'multi_match': {
                    'query': consulta,
                    'fields': ['nombre_proceso^2', 'descripcion']
                }
            }]
        }
    },
    'highlight': {'fields': {'nombre_proceso': {}, 'descripcion': {}}},
    'size': 5
}

resp = client.search(index=INDEX_NAME, **query)
filas = []
for h in resp['hits']['hits']:
    s = h['_source']
    filas.append({
        'rank': len(filas)+1,
        'id_proceso': s.get('id_proceso'),
        'score': h.get('_score'),
        'nombre_proceso': s.get('nombre_proceso'),
        'highlight': ' ... '.join(sum(h.get('highlight', {}).values(), [])),
        'url_secop': s.get('url_secop')
    })
resultados = pd.DataFrame(filas)
resultados

## 7. Experimento controlado

Cambia una sola cosa: la consulta, el filtro o el peso del campo. No cambies todo al tiempo.

In [ ]:
consulta_b = 'blindajes aeronáuticos'
query['query']['bool']['must'][0]['multi_match']['query'] = consulta_b
resp_b = client.search(index=INDEX_NAME, **query)
[(h['_score'], h['_source']['id_proceso'], h['_source']['nombre_proceso']) for h in resp_b['hits']['hits']]

## 8. Guardar evidencia

El entregable no es solo que corra. Debes dejar una evidencia que diga qué buscaste, qué apareció y qué límite tiene la interpretación.

In [ ]:
from pathlib import Path

resultados.to_csv('s07_elastic_resultados.csv', index=False, encoding='utf-8-sig')

hito = f'''# Hito S07B — Despliegue Elasticsearch

- Índice: {INDEX_NAME}
- Consulta A: {consulta}
- Consulta B: {consulta_b}
- Documentos cargados: {len(corpus)}

## Top 5 consulta A

{resultados.to_markdown(index=False)}

## Interpretación
Escribe aquí qué proceso leerías primero y por qué.

## Límite
Un score alto indica relevancia textual respecto a la consulta; no demuestra irregularidad ni riesgo.
'''
Path('hito_s07b_despliegue.md').write_text(hito, encoding='utf-8')
print(hito)

## 9. Cerrar o limpiar

Si estás usando un proyecto personal de prueba, no olvides revisar costos, borrar recursos que no necesites o detener el deployment según corresponda a tu plan.